In [7]:
import os, subprocess
print(os.environ["JAVA_HOME"])
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)
print(os.path.exists(os.path.join(os.environ["JAVA_HOME"], "bin", "java.exe")))

C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot\
java version "1.8.0_511"
Java(TM) SE Runtime Environment (build 1.8.0_511-b33)
Java HotSpot(TM) 64-Bit Server VM (build 25.511-b33, mixed mode)

True


In [8]:
import os, subprocess
jh = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot"
print(subprocess.run([jh + r"\bin\java.exe", "-version"], capture_output=True, text=True).stderr)

openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)



In [1]:
!java -version

openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)


In [4]:
import os, sys
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# work around the unreadable computer name
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

In [6]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("RetailAnalyticsLab")
         .master("local[*]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())
print(spark.version)
spark.range(5).show()

4.0.1
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [7]:
import os, sys
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost" 

In [8]:
spark = (SparkSession.builder
         .appName("RetailAnalyticsLab")
         .master("local[*]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())

In [9]:
from pyspark.sql import SparkSession 
spark = (SparkSession.builder 
        .appName("RetailAnalyticsLab") 
        .master("local[*]") 
        .getOrCreate()) 
print("Spark version:", spark.version) 
print("Application:", spark.sparkContext.appName) 

Spark version: 4.0.1
Application: RetailAnalyticsLab


In [10]:
sales = (spark.read 
         .option("header", True) 
         .option("inferSchema", True) 
         .csv("sales_transactions.csv")) 
 
products = (spark.read 
            .option("header", True) 
            .option("inferSchema", True) 
            .csv("products.csv")) 
 
stores = (spark.read 
          .option("header", True) 
          .option("inferSchema", True) 
          .csv("stores.csv")) 

In [11]:
sales.printSchema() 
sales.show(5, truncate=False) 
products.show(5, truncate=False) 
stores.show(5, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_rate: double (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)

+--------------+----------------+-----------+----------+--------+--------+----------+-------------+------------+---------------+---------+
|transaction_id|transaction_date|customer_id|product_id|store_id|quantity|unit_price|discount_rate|sales_amount|payment_method |status   |
+--------------+----------------+-----------+----------+--------+--------+----------+-------------+------------+---------------+---------+
|T00001        |2025-11-24      |C0023      |P009      |S02     |1       |484.03    |0.15         |411.4

In [16]:
sales.createOrReplaceTempView("sales") 
products.createOrReplaceTempView("products") 
stores.createOrReplaceTempView("stores")
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         | products|       true|
|         |    sales|       true|
|         |   stores|       true|
+---------+---------+-----------+



In [17]:
#Exercise 2 — Select specific columns

result = spark.sql(""" 
SELECT * 
FROM sales 
""") 
result.show()

+--------------+----------------+-----------+----------+--------+--------+----------+-------------+------------+---------------+---------+
|transaction_id|transaction_date|customer_id|product_id|store_id|quantity|unit_price|discount_rate|sales_amount| payment_method|   status|
+--------------+----------------+-----------+----------+--------+--------+----------+-------------+------------+---------------+---------+
|        T00001|      2025-11-24|      C0023|      P009|     S02|       1|    484.03|         0.15|      411.43|Online Transfer|Completed|
|        T00002|      2025-10-30|      C0167|      P001|     S01|       1|     26.42|          0.1|       23.78|           Cash|Completed|
|        T00003|      2025-12-26|      C0040|      P008|     S04|       2|     129.0|          0.0|       258.0|Online Transfer|Completed|
|        T00004|      2025-04-21|      C0118|      P003|     S04|       1|     60.69|          0.0|       60.69|           Card|Completed|
|        T00005|      2025-

In [18]:
#Exercise 1 — Display every columne

result = spark.sql(""" 
SELECT * 
FROM sales 
LIMIT 10; 
""") 
result.show(truncate=False)

+--------------+----------------+-----------+----------+--------+--------+----------+-------------+------------+---------------+---------+
|transaction_id|transaction_date|customer_id|product_id|store_id|quantity|unit_price|discount_rate|sales_amount|payment_method |status   |
+--------------+----------------+-----------+----------+--------+--------+----------+-------------+------------+---------------+---------+
|T00001        |2025-11-24      |C0023      |P009      |S02     |1       |484.03    |0.15         |411.43      |Online Transfer|Completed|
|T00002        |2025-10-30      |C0167      |P001      |S01     |1       |26.42     |0.1          |23.78       |Cash           |Completed|
|T00003        |2025-12-26      |C0040      |P008      |S04     |2       |129.0     |0.0          |258.0       |Online Transfer|Completed|
|T00004        |2025-04-21      |C0118      |P003      |S04     |1       |60.69     |0.0          |60.69       |Card           |Completed|
|T00005        |2025-10-02 

In [19]:
Exercise 2 — Select specific columns 

result = spark.sql(""" 
SELECT transaction_id, transaction_date, product_id, quantity, sales_amount 
FROM sales 
LIMIT 15; 
""") 
result.show(truncate=False)

+--------------+----------------+----------+--------+------------+
|transaction_id|transaction_date|product_id|quantity|sales_amount|
+--------------+----------------+----------+--------+------------+
|T00001        |2025-11-24      |P009      |1       |411.43      |
|T00002        |2025-10-30      |P001      |1       |23.78       |
|T00003        |2025-12-26      |P008      |2       |258.0       |
|T00004        |2025-04-21      |P003      |1       |60.69       |
|T00005        |2025-10-02      |P003      |3       |160.65      |
|T00006        |2025-12-05      |P003      |3       |169.57      |
|T00007        |2025-07-09      |P009      |2       |1017.96     |
|T00008        |2025-05-19      |P011      |1       |79.0        |
|T00009        |2025-06-11      |P013      |1       |8.9         |
|T00010        |2025-08-08      |P012      |5       |63.75       |
|T00011        |2025-03-20      |P014      |1       |16.2        |
|T00012        |2025-12-15      |P018      |2       |67.9     

In [20]:
Exercise 3 — Explore the product catalogue

result = spark.sql(""" 
SELECT product_name, category, list_price 
FROM products; 
""") 
result.show(truncate=False)

+-------------------+-----------+----------+
|product_name       |category   |list_price|
+-------------------+-----------+----------+
|Wireless Mouse     |Accessories|25.9      |
|Mechanical Keyboard|Accessories|89.0      |
|USB-C Hub          |Accessories|59.5      |
|Laptop Stand       |Accessories|42.0      |
|27-inch Monitor    |Electronics|699.0     |
|External SSD 1TB   |Electronics|389.0     |
|Wireless Headphones|Electronics|249.0     |
|Webcam HD          |Electronics|129.0     |
|Office Chair       |Furniture  |499.0     |
|Standing Desk      |Furniture  |1250.0    |
|Desk Lamp          |Furniture  |79.0      |
|Notebook Set       |Stationery |12.5      |
|Ballpoint Pens     |Stationery |8.9       |
|A4 Paper Pack      |Stationery |18.0      |
|Whiteboard Markers |Stationery |15.5      |
|Water Bottle       |Lifestyle  |22.0      |
|Backpack           |Lifestyle  |119.0     |
|Travel Mug         |Lifestyle  |35.0      |
|Bluetooth Speaker  |Electronics|179.0     |
|Laptop Sl

In [21]:
#Exercise 4 — Explore store locations

result = spark.sql(""" 
SELECT store_id, store_name, district 
FROM stores; 
""") 
result.show(truncate=False)

+--------+-------------------+------------+
|store_id|store_name         |district    |
+--------+-------------------+------------+
|S01     |Bandar Seri Begawan|Brunei-Muara|
|S02     |Kuala Belait       |Belait      |
|S03     |Tutong             |Tutong      |
|S04     |Seria              |Belait      |
+--------+-------------------+------------+



In [43]:
#Exercise 5 — One condition: completed transactions 

result = spark.sql(""" 
SELECT transaction_id, transaction_date, sales_amount, status 
FROM sales 
WHERE status = 'Completed' 
LIMIT 20; 
""") 
result.show(truncate=False)

+--------------+----------------+------------+---------+
|transaction_id|transaction_date|sales_amount|status   |
+--------------+----------------+------------+---------+
|T00001        |2025-11-24      |411.43      |Completed|
|T00002        |2025-10-30      |23.78       |Completed|
|T00003        |2025-12-26      |258.0       |Completed|
|T00004        |2025-04-21      |60.69       |Completed|
|T00005        |2025-10-02      |160.65      |Completed|
|T00006        |2025-12-05      |169.57      |Completed|
|T00007        |2025-07-09      |1017.96     |Completed|
|T00008        |2025-05-19      |79.0        |Completed|
|T00009        |2025-06-11      |8.9         |Completed|
|T00010        |2025-08-08      |63.75       |Completed|
|T00012        |2025-12-15      |67.9        |Completed|
|T00013        |2025-05-15      |364.14      |Completed|
|T00014        |2025-10-04      |49.0        |Completed|
|T00015        |2025-05-03      |164.95      |Completed|
|T00016        |2025-03-07     

In [22]:
Exercise 6 — One condition: high-value transactions

result = spark.sql(""" 
SELECT transaction_id, customer_id, sales_amount 
FROM sales 
WHERE sales_amount > 300; 
""") 
result.show(truncate=False)

+--------------+-----------+------------+
|transaction_id|customer_id|sales_amount|
+--------------+-----------+------------+
|T00001        |C0023      |411.43      |
|T00007        |C0098      |1017.96     |
|T00013        |C0042      |364.14      |
|T00019        |C0122      |508.98      |
|T00025        |C0081      |712.98      |
|T00030        |C0168      |991.95      |
|T00033        |C0177      |898.2       |
|T00053        |C0155      |3633.75     |
|T00054        |C0043      |474.05      |
|T00058        |C0056      |498.0       |
|T00060        |C0161      |1062.5      |
|T00063        |C0152      |700.2       |
|T00065        |C0071      |389.0       |
|T00069        |C0073      |507.96      |
|T00074        |C0054      |1091.25     |
|T00080        |C0072      |998.0       |
|T00082        |C0083      |3273.75     |
|T00084        |C0156      |1896.2      |
|T00085        |C0119      |2500.0      |
|T00094        |C0163      |531.8       |
+--------------+-----------+------

In [23]:
#Exercise 7 — One condition: payment method

result = spark.sql(""" 
SELECT transaction_id, payment_method, sales_amount, status 
FROM sales 
WHERE payment_method = 'E-Wallet';
""") 
result.show(truncate=False)

+--------------+--------------+------------+---------+
|transaction_id|payment_method|sales_amount|status   |
+--------------+--------------+------------+---------+
|T00014        |E-Wallet      |49.0        |Completed|
|T00015        |E-Wallet      |164.95      |Completed|
|T00016        |E-Wallet      |31.5        |Completed|
|T00020        |E-Wallet      |39.9        |Completed|
|T00022        |E-Wallet      |46.5        |Completed|
|T00026        |E-Wallet      |103.89      |Completed|
|T00036        |E-Wallet      |51.8        |Completed|
|T00037        |E-Wallet      |26.7        |Completed|
|T00047        |E-Wallet      |51.8        |Completed|
|T00048        |E-Wallet      |40.7        |Completed|
|T00052        |E-Wallet      |217.38      |Completed|
|T00055        |E-Wallet      |8.46        |Completed|
|T00062        |E-Wallet      |77.7        |Completed|
|T00069        |E-Wallet      |507.96      |Completed|
|T00077        |E-Wallet      |15.3        |Completed|
|T00085   

In [24]:
#Exercise 8 — One condition: product category

result = spark.sql(""" 
SELECT product_id, product_name, category, list_price 
FROM products 
WHERE category = 'Electronics'; 
""") 
result.show(truncate=False)

+----------+-------------------+-----------+----------+
|product_id|product_name       |category   |list_price|
+----------+-------------------+-----------+----------+
|P005      |27-inch Monitor    |Electronics|699.0     |
|P006      |External SSD 1TB   |Electronics|389.0     |
|P007      |Wireless Headphones|Electronics|249.0     |
|P008      |Webcam HD          |Electronics|129.0     |
|P019      |Bluetooth Speaker  |Electronics|179.0     |
+----------+-------------------+-----------+----------+



In [44]:
#Exercise 9 — Multiple conditions with AND 

result = spark.sql(""" 
SELECT transaction_id, status, sales_amount 
FROM sales 
WHERE status = 'Completed' 
  AND sales_amount > 200; 
""") 
result.show(truncate=False)

+--------------+---------+------------+
|transaction_id|status   |sales_amount|
+--------------+---------+------------+
|T00001        |Completed|411.43      |
|T00003        |Completed|258.0       |
|T00007        |Completed|1017.96     |
|T00013        |Completed|364.14      |
|T00019        |Completed|508.98      |
|T00025        |Completed|712.98      |
|T00028        |Completed|236.55      |
|T00033        |Completed|898.2       |
|T00038        |Completed|249.0       |
|T00051        |Completed|219.3       |
|T00052        |Completed|217.38      |
|T00053        |Completed|3633.75     |
|T00054        |Completed|474.05      |
|T00058        |Completed|498.0       |
|T00060        |Completed|1062.5      |
|T00063        |Completed|700.2       |
|T00065        |Completed|389.0       |
|T00069        |Completed|507.96      |
|T00073        |Completed|250.0       |
|T00074        |Completed|1091.25     |
+--------------+---------+------------+
only showing top 20 rows


In [25]:
#Exercise 10 — Multiple conditions with OR

result = spark.sql(""" 
SELECT transaction_id, status, sales_amount 
FROM sales 
WHERE status = 'Completed' 
  AND sales_amount > 200; 
""") 
result.show(truncate=False)

+--------------+---------+------------+
|transaction_id|status   |sales_amount|
+--------------+---------+------------+
|T00001        |Completed|411.43      |
|T00003        |Completed|258.0       |
|T00007        |Completed|1017.96     |
|T00013        |Completed|364.14      |
|T00019        |Completed|508.98      |
|T00025        |Completed|712.98      |
|T00028        |Completed|236.55      |
|T00033        |Completed|898.2       |
|T00038        |Completed|249.0       |
|T00051        |Completed|219.3       |
|T00052        |Completed|217.38      |
|T00053        |Completed|3633.75     |
|T00054        |Completed|474.05      |
|T00058        |Completed|498.0       |
|T00060        |Completed|1062.5      |
|T00063        |Completed|700.2       |
|T00065        |Completed|389.0       |
|T00069        |Completed|507.96      |
|T00073        |Completed|250.0       |
|T00074        |Completed|1091.25     |
+--------------+---------+------------+
only showing top 20 rows


In [26]:
#Exercise 11 — Combine AND and OR with parentheses

result = spark.sql(""" 
SELECT transaction_id, payment_method, sales_amount 
FROM sales 
WHERE payment_method = 'Cash' 
   OR payment_method = 'Card'; 
""") 
result.show(truncate=False)

+--------------+--------------+------------+
|transaction_id|payment_method|sales_amount|
+--------------+--------------+------------+
|T00002        |Cash          |23.78       |
|T00004        |Card          |60.69       |
|T00005        |Card          |160.65      |
|T00006        |Card          |169.57      |
|T00008        |Card          |79.0        |
|T00010        |Card          |63.75       |
|T00011        |Card          |16.2        |
|T00012        |Card          |67.9        |
|T00017        |Cash          |8.01        |
|T00018        |Card          |147.59      |
|T00019        |Card          |508.98      |
|T00021        |Card          |26.7        |
|T00023        |Card          |56.52       |
|T00024        |Card          |33.82       |
|T00027        |Cash          |85.68       |
|T00029        |Card          |49.98       |
|T00031        |Card          |42.84       |
|T00033        |Card          |898.2       |
|T00034        |Card          |12.5        |
|T00038   

In [27]:
#Exercise 12 — Range filtering with BETWEEN

result = spark.sql(""" 
SELECT transaction_id, payment_method, status, sales_amount 
FROM sales 
WHERE status = 'Completed' 
  AND (payment_method = 'Cash' OR payment_method = 'Card'); 
""")
result.show(truncate=False)

+--------------+--------------+---------+------------+
|transaction_id|payment_method|status   |sales_amount|
+--------------+--------------+---------+------------+
|T00002        |Cash          |Completed|23.78       |
|T00004        |Card          |Completed|60.69       |
|T00005        |Card          |Completed|160.65      |
|T00006        |Card          |Completed|169.57      |
|T00008        |Card          |Completed|79.0        |
|T00010        |Card          |Completed|63.75       |
|T00012        |Card          |Completed|67.9        |
|T00017        |Cash          |Completed|8.01        |
|T00018        |Card          |Completed|147.59      |
|T00019        |Card          |Completed|508.98      |
|T00021        |Card          |Completed|26.7        |
|T00023        |Card          |Completed|56.52       |
|T00027        |Cash          |Completed|85.68       |
|T00029        |Card          |Completed|49.98       |
|T00031        |Card          |Completed|42.84       |
|T00033   

In [28]:
#Exercise 13 — Matching a set of values with IN

result = spark.sql(""" 
SELECT product_name, category, list_price 
FROM products 
WHERE list_price BETWEEN 50 AND 150; 
""") 
result.show(truncate=False)

+-------------------+-----------+----------+
|product_name       |category   |list_price|
+-------------------+-----------+----------+
|Mechanical Keyboard|Accessories|89.0      |
|USB-C Hub          |Accessories|59.5      |
|Webcam HD          |Electronics|129.0     |
|Desk Lamp          |Furniture  |79.0      |
|Backpack           |Lifestyle  |119.0     |
+-------------------+-----------+----------+



In [29]:
#Exercise 14 — Pattern matching with LIKE

result = spark.sql(""" 
SELECT transaction_id, status, sales_amount 
FROM sales 
WHERE status IN ('Returned', 'Cancelled'); 
""") 
result.show(truncate=False)

+--------------+---------+------------+
|transaction_id|status   |sales_amount|
+--------------+---------+------------+
|T00011        |Cancelled|16.2        |
|T00024        |Returned |33.82       |
|T00030        |Cancelled|991.95      |
|T00078        |Returned |46.55       |
|T00084        |Cancelled|1896.2      |
|T00086        |Cancelled|22.0        |
|T00110        |Returned |12.5        |
|T00133        |Returned |59.5        |
|T00135        |Cancelled|328.64      |
|T00170        |Cancelled|19.8        |
|T00180        |Returned |677.33      |
|T00200        |Cancelled|1147.5      |
|T00213        |Cancelled|230.86      |
|T00227        |Returned |79.0        |
|T00275        |Cancelled|103.6       |
|T00284        |Cancelled|36.67       |
|T00285        |Returned |152.15      |
|T00303        |Returned |358.46      |
|T00325        |Cancelled|15.53       |
|T00337        |Returned |38.56       |
+--------------+---------+------------+
only showing top 20 rows


In [30]:
#Exercise 15 — Ascending order

result = spark.sql(""" 
SELECT product_id, product_name, category 
FROM products 
WHERE product_name LIKE '%Wireless%'; 
""") 
result.show(truncate=False)

+----------+-------------------+-----------+
|product_id|product_name       |category   |
+----------+-------------------+-----------+
|P001      |Wireless Mouse     |Accessories|
|P007      |Wireless Headphones|Electronics|
+----------+-------------------+-----------+



In [31]:
#Exercise 16 — Descending order

result = spark.sql(""" 
SELECT product_name, category, list_price 
FROM products 
ORDER BY list_price ASC; 
""")
result.show(truncate=False)

+-------------------+-----------+----------+
|product_name       |category   |list_price|
+-------------------+-----------+----------+
|Ballpoint Pens     |Stationery |8.9       |
|Notebook Set       |Stationery |12.5      |
|Whiteboard Markers |Stationery |15.5      |
|A4 Paper Pack      |Stationery |18.0      |
|Water Bottle       |Lifestyle  |22.0      |
|Wireless Mouse     |Accessories|25.9      |
|Travel Mug         |Lifestyle  |35.0      |
|Laptop Stand       |Accessories|42.0      |
|Laptop Sleeve      |Accessories|49.0      |
|USB-C Hub          |Accessories|59.5      |
|Desk Lamp          |Furniture  |79.0      |
|Mechanical Keyboard|Accessories|89.0      |
|Backpack           |Lifestyle  |119.0     |
|Webcam HD          |Electronics|129.0     |
|Bluetooth Speaker  |Electronics|179.0     |
|Wireless Headphones|Electronics|249.0     |
|External SSD 1TB   |Electronics|389.0     |
|Office Chair       |Furniture  |499.0     |
|27-inch Monitor    |Electronics|699.0     |
|Standing 

In [32]:
#Exercise 17 — Sort by more than one column 

result = spark.sql(""" 
SELECT product_name, category, list_price 
FROM products 
ORDER BY list_price DESC; 
""") 
result.show(truncate=False)

+-------------------+-----------+----------+
|product_name       |category   |list_price|
+-------------------+-----------+----------+
|Standing Desk      |Furniture  |1250.0    |
|27-inch Monitor    |Electronics|699.0     |
|Office Chair       |Furniture  |499.0     |
|External SSD 1TB   |Electronics|389.0     |
|Wireless Headphones|Electronics|249.0     |
|Bluetooth Speaker  |Electronics|179.0     |
|Webcam HD          |Electronics|129.0     |
|Backpack           |Lifestyle  |119.0     |
|Mechanical Keyboard|Accessories|89.0      |
|Desk Lamp          |Furniture  |79.0      |
|USB-C Hub          |Accessories|59.5      |
|Laptop Sleeve      |Accessories|49.0      |
|Laptop Stand       |Accessories|42.0      |
|Travel Mug         |Lifestyle  |35.0      |
|Wireless Mouse     |Accessories|25.9      |
|Water Bottle       |Lifestyle  |22.0      |
|A4 Paper Pack      |Stationery |18.0      |
|Whiteboard Markers |Stationery |15.5      |
|Notebook Set       |Stationery |12.5      |
|Ballpoint

In [33]:
#Exercise 18 — Sort transactions by date

result = spark.sql(""" 
SELECT product_name, category, list_price 
FROM products 
ORDER BY category ASC, list_price DESC; 
""") 
result.show(truncate=False)

+-------------------+-----------+----------+
|product_name       |category   |list_price|
+-------------------+-----------+----------+
|Mechanical Keyboard|Accessories|89.0      |
|USB-C Hub          |Accessories|59.5      |
|Laptop Sleeve      |Accessories|49.0      |
|Laptop Stand       |Accessories|42.0      |
|Wireless Mouse     |Accessories|25.9      |
|27-inch Monitor    |Electronics|699.0     |
|External SSD 1TB   |Electronics|389.0     |
|Wireless Headphones|Electronics|249.0     |
|Bluetooth Speaker  |Electronics|179.0     |
|Webcam HD          |Electronics|129.0     |
|Standing Desk      |Furniture  |1250.0    |
|Office Chair       |Furniture  |499.0     |
|Desk Lamp          |Furniture  |79.0      |
|Backpack           |Lifestyle  |119.0     |
|Travel Mug         |Lifestyle  |35.0      |
|Water Bottle       |Lifestyle  |22.0      |
|A4 Paper Pack      |Stationery |18.0      |
|Whiteboard Markers |Stationery |15.5      |
|Notebook Set       |Stationery |12.5      |
|Ballpoint

In [34]:
#Exercise 19 — Top five completed transactions

result = spark.sql(""" 
SELECT transaction_id, transaction_date, customer_id, sales_amount 
FROM sales 
ORDER BY transaction_date DESC 
LIMIT 15; 
""") 
result.show(truncate=False)

+--------------+----------------+-----------+------------+
|transaction_id|transaction_date|customer_id|sales_amount|
+--------------+----------------+-----------+------------+
|T00067        |2025-12-31      |C0122      |24.6        |
|T00300        |2025-12-31      |C0022      |240.3       |
|T00271        |2025-12-31      |C0020      |7.57        |
|T00628        |2025-12-31      |C0082      |991.95      |
|T00918        |2025-12-31      |C0074      |76.63       |
|T00974        |2025-12-31      |C0110      |238.0       |
|T01101        |2025-12-31      |C0136      |2097.0      |
|T01149        |2025-12-31      |C0090      |33.95       |
|T00390        |2025-12-30      |C0052      |18.0        |
|T01031        |2025-12-30      |C0057      |69.26       |
|T00063        |2025-12-29      |C0152      |700.2       |
|T00915        |2025-12-28      |C0038      |249.0       |
|T01148        |2025-12-28      |C0072      |1125.0      |
|T00326        |2025-12-27      |C0179      |389.0      

In [35]:
#Exercise 20 — Lowest-value completed transactions

result = spark.sql(""" 
SELECT transaction_id, transaction_date, customer_id, sales_amount 
FROM sales 
WHERE status = 'Completed' 
ORDER BY sales_amount DESC 
LIMIT 5; 
""")
result.show(truncate=False)

+--------------+----------------+-----------+------------+
|transaction_id|transaction_date|customer_id|sales_amount|
+--------------+----------------+-----------+------------+
|T00982        |2025-07-15      |C0106      |6062.5      |
|T00171        |2025-11-10      |C0148      |5312.5      |
|T01035        |2025-08-10      |C0075      |4750.0      |
|T00624        |2025-09-24      |C0166      |4500.0      |
|T00731        |2025-12-03      |C0042      |4250.0      |
+--------------+----------------+-----------+------------+



In [36]:
#Exercise 21 — Create a calculated column 

result = spark.sql(""" 
SELECT transaction_id, quantity, unit_price, 
       ROUND(quantity * unit_price, 2) AS gross_value, 
       sales_amount 
FROM sales 
LIMIT 15; 
""") 
result.show(truncate=False)

+--------------+--------+----------+-----------+------------+
|transaction_id|quantity|unit_price|gross_value|sales_amount|
+--------------+--------+----------+-----------+------------+
|T00001        |1       |484.03    |484.03     |411.43      |
|T00002        |1       |26.42     |26.42      |23.78       |
|T00003        |2       |129.0     |258.0      |258.0       |
|T00004        |1       |60.69     |60.69      |60.69       |
|T00005        |3       |59.5      |178.5      |160.65      |
|T00006        |3       |59.5      |178.5      |169.57      |
|T00007        |2       |508.98    |1017.96    |1017.96     |
|T00008        |1       |79.0      |79.0       |79.0        |
|T00009        |1       |8.9       |8.9        |8.9         |
|T00010        |5       |12.75     |63.75      |63.75       |
|T00011        |1       |18.0      |18.0       |16.2        |
|T00012        |2       |33.95     |67.9       |67.9        |
|T00013        |3       |121.38    |364.14     |364.14      |
|T00014 

In [37]:
#Exercise 22 — Basic aggregate functions

result = spark.sql(""" 
SELECT COUNT(*) AS transaction_count, 
       ROUND(SUM(sales_amount), 2) AS recorded_sales, 
       ROUND(AVG(sales_amount), 2) AS average_transaction, 
       ROUND(MIN(sales_amount), 2) AS minimum_transaction, 
       ROUND(MAX(sales_amount), 2) AS maximum_transaction 
FROM sales; 
""") 
result.show(truncate=False)

+-----------------+--------------+-------------------+-------------------+-------------------+
|transaction_count|recorded_sales|average_transaction|minimum_transaction|maximum_transaction|
+-----------------+--------------+-------------------+-------------------+-------------------+
|1200             |399302.21     |332.75             |7.34               |6062.5             |
+-----------------+--------------+-------------------+-------------------+-------------------+



In [38]:
#Exercise 23 — Count records by status

result = spark.sql(""" 
SELECT status, COUNT(*) AS transaction_count 
FROM sales 
GROUP BY status 
ORDER BY transaction_count DESC; 
""") 
result.show(truncate=False)

+---------+-----------------+
|status   |transaction_count|
+---------+-----------------+
|Completed|1138             |
|Cancelled|34               |
|Returned |28               |
+---------+-----------------+



In [42]:
#Exercise 24 — Payment method profile

result = spark.sql(""" 
SELECT payment_method, 
       COUNT(*) AS transaction_count, 
       ROUND(AVG(sales_amount), 2) AS average_amount 
FROM sales 
GROUP BY payment_method 
ORDER BY transaction_count DESC; 
""") 
result.show(truncate=False)

+---------------+-----------------+--------------+
|payment_method |transaction_count|average_amount|
+---------------+-----------------+--------------+
|Card           |553              |316.56        |
|Online Transfer|243              |403.56        |
|E-Wallet       |231              |317.13        |
|Cash           |173              |305.91        |
+---------------+-----------------+--------------+



In [41]:
#EXERCSE 25 - HAVING: filter aggregated groups

result = spark.sql(""" 
SELECT store_id, ROUND(SUM(sales_amount), 2) AS total_sales 
FROM sales 
WHERE status = 'Completed' 
GROUP BY store_id 
HAVING SUM(sales_amount) > 100000 
ORDER BY total_sales DESC; 
""") 
result.show(truncate=False)

+--------+-----------+
|store_id|total_sales|
+--------+-----------+
|S04     |113821.87  |
|S01     |103964.78  |
+--------+-----------+

